# 03 – Exploratory Data Analysis (EDA)

Deep-dive into price behaviour, returns, volatility, and inter-stock correlation.

Topics covered:
1. Descriptive statistics
2. Normalised price comparison
3. Rolling volatility comparison
4. Correlation matrix
5. Cumulative returns
6. Monthly seasonality
7. Drawdown analysis
8. Risk vs. Return scatter

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.preprocessing import load_processed
from src.analysis import PortfolioMetrics, correlation_matrix

TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']
data = {t: load_processed(t) for t in TICKERS if
        os.path.exists(f'../data/processed/{t}_processed.csv')}

print(f'Loaded: {list(data.keys())}')

## 3.1 Descriptive Statistics

In [ ]:
stats = pd.DataFrame({
    t: {
        'Mean Close'      : df['close'].mean(),
        'Std Close'       : df['close'].std(),
        'Min'             : df['close'].min(),
        'Max'             : df['close'].max(),
        'Mean Daily Ret'  : df['daily_return'].mean(),
        'Std Daily Ret'   : df['daily_return'].std(),
        'Skewness'        : df['daily_return'].skew(),
        'Kurtosis'        : df['daily_return'].kurt(),
    }
    for t, df in data.items()
}).T.round(4)

stats

## 3.2 Normalised Price Comparison (Base = 100)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for ticker, df in data.items():
    norm = df['close'] / df['close'].iloc[0] * 100
    ax.plot(df.index, norm, label=ticker, linewidth=1.5)

ax.axhline(100, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('Normalised Price (Base = 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Normalised Price')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/normalised_price.png', dpi=150)
plt.show()

## 3.3 Rolling 21-Day Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker, df in data.items():
    ax.plot(df.index, df['volatility_21d'] * 100, label=ticker, linewidth=1.2)

ax.set_title('21-Day Rolling Annualised Volatility (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/rolling_volatility.png', dpi=150)
plt.show()

## 3.4 Correlation Matrix

In [ ]:
from src.analysis import correlation_matrix
from src.visualization import StockVisualizer

corr = correlation_matrix(data, col='close')
StockVisualizer.plot_correlation(corr, save=True)

## 3.5 Cumulative Returns

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for ticker, df in data.items():
    cum = (1 + df['daily_return']).cumprod() - 1
    ax.plot(df.index, cum * 100, label=ticker, linewidth=1.5)

ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('Cumulative Returns (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Return (%)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/cumulative_returns.png', dpi=150)
plt.show()

## 3.6 Monthly Seasonality (AAPL)

In [ ]:
aapl = data['AAPL'].copy()
aapl['month'] = aapl.index.month
aapl['month_name'] = aapl.index.month_name()

monthly = aapl.groupby('month')['daily_return'].mean() * 100
monthly.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

colors = ['#26a69a' if v >= 0 else '#ef5350' for v in monthly]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly.index, monthly.values, color=colors, edgecolor='none')
ax.axhline(0, color='white', linewidth=0.8)
ax.set_title('AAPL – Average Monthly Return (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Daily Return (%)')
plt.tight_layout()
plt.savefig('../images/charts/AAPL_monthly_seasonality.png', dpi=150)
plt.show()

## 3.7 Drawdown Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker, df in data.items():
    cum = (1 + df['daily_return']).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak * 100
    ax.plot(df.index, drawdown, label=ticker, linewidth=1.2)

ax.fill_between(list(data.values())[0].index,
                (list(data.values())[0]['daily_return'] * 0),
                0, alpha=0.05, color='white')
ax.set_title('Drawdown from Peak (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/drawdown.png', dpi=150)
plt.show()

## 3.8 Risk vs. Return Scatter

In [ ]:
risk_return = []
for ticker, df in data.items():
    m = PortfolioMetrics(df['daily_return'])
    risk_return.append({
        'Ticker': ticker,
        'Ann. Return (%)': m.annualized_return() * 100,
        'Ann. Volatility (%)': m.annualized_volatility() * 100,
        'Sharpe': m.sharpe_ratio(),
    })

rr_df = pd.DataFrame(risk_return)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    rr_df['Ann. Volatility (%)'], rr_df['Ann. Return (%)'],
    c=rr_df['Sharpe'], cmap='RdYlGn', s=200, edgecolors='white', zorder=5
)
for _, row in rr_df.iterrows():
    ax.annotate(row['Ticker'], (row['Ann. Volatility (%)'], row['Ann. Return (%)']),
                textcoords='offset points', xytext=(8, 4), fontsize=10)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
ax.set_title('Risk vs. Return', fontsize=14, fontweight='bold')
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
plt.tight_layout()
plt.savefig('../images/charts/risk_vs_return.png', dpi=150)
plt.show()

rr_df.set_index('Ticker').round(2)

---
**Next →** `04_technical_analysis.ipynb`